# Task 1. Dataset and Dependencies

### 1.1. Install and import all required dependencies

In [1]:
%pip install -q pgserver==0.1.4 psycopg2-binary sqlalchemy pandas pyarrow requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 50.2 MB/s eta 0:00:00


In [2]:
import math
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
import pyarrow.parquet as pq
import pgserver
from sqlalchemy import create_engine, text
from IPython.display import display

Start and check server

In [3]:
pg = pgserver.get_server(
    "/tmp/hw7_pg",
    cleanup_mode="stop"
)

engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


Add helper function to execute SQL without expected output to show (e.g. create table)

In [4]:
def run_sql(query):
    with engine.begin() as conn:
        conn.execute(text(query))
        #conn.exec_driver_sql(query)

Add helper function to execute SQL queries and show output (e.g., SELECT query)

In [5]:
def run_query(query, params=None):
    with engine.connect() as conn:
        return pd.read_sql_query(
            text(query), conn, params=params)

### 1.2. Download and Sample the March 2024 Parquet

The file is downloaded from the official NYC TLC archive. We take approximately 400,000-450,000 observations spaced across the full file using a fixed sampling stride. This keeps observations across March and makes the sampling reproducible. Only the columns needed for the homework are loaded into memory per batch.

In [6]:
URL = (
    'https://d37ci6vzurychx.cloudfront.net/trip-data/'
    'yellow_tripdata_2024-03.parquet'
)
parquet_path = Path('/content/yellow_tripdata_2024-03.parquet')
if not parquet_path.exists():
    urlretrieve(URL, str(parquet_path))

columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]

parquet_file = pq.ParquetFile(str(parquet_path))
source_rows = parquet_file.metadata.num_rows
stride = max(1, math.ceil(source_rows / 450_000))

parts = [
    batch.to_pandas().iloc[::stride].copy()
    for batch in parquet_file.iter_batches(batch_size=100_000, columns=columns)
]
df = pd.concat(parts, ignore_index=True)
del parts

print(f'Parquet rows: {source_rows:,}; sampling stride: {stride}')
print(f'Sampled rows before cleaning: {len(df):,}')

Parquet rows: 3,582,628; sampling stride: 8
Sampled rows before cleaning: 447,829


### 1.3. Data Cleaning and Raw PostgreSQL Table

Keep only pickups from March 2024. Drop rows with invalid fares, distances or missing pickup zones. We preserve the local timestamps as `TIMESTAMP WITHOUT TIME ZONE` in the raw table.

In [7]:
march_start = pd.Timestamp('2024-03-01')
april_start = pd.Timestamp('2024-04-01')

mask = (
    (df['tpep_pickup_datetime'] >= march_start)
    & (df['tpep_pickup_datetime'] < april_start)
    & (df['fare_amount'] > 0)
    & (df['fare_amount'] < 500)
    & (df['trip_distance'] > 0)
    & (df['trip_distance'] < 100)
    & df['PULocationID'].notna()
)
df = df.loc[mask].copy()

df.to_sql(
    'nyc_taxi_raw',
    engine,
    if_exists='replace',
    index=False,
    chunksize=10_000,
)

print(f'Clean sample loaded: {len(df):,} trips')
print('Pickup date range:', df['tpep_pickup_datetime'].min(),
      'to', df['tpep_pickup_datetime'].max())
del df

Clean sample loaded: 430,007 trips
Pickup date range: 2024-03-01 00:00:00 to 2024-03-31 23:59:47


In [8]:
display(run_query("""
SELECT
    COUNT(*) AS n_trips,
    COUNT(DISTINCT "PULocationID") AS n_pickup_zones,
    MIN(tpep_pickup_datetime)::DATE AS first_day,
    MAX(tpep_pickup_datetime)::DATE AS last_day
FROM nyc_taxi_raw;
"""))

,n_trips,n_pickup_zones,first_day,last_day
0,430007,245,2024-03-01,2024-03-31


### 1.4. Daily Aggregation with Calendar Spine

The dense `t7_hw_taxi_daily` table contains one record per **pickup zone * calendar day** in March. Missing zone-days are represented by `trip_count = 0` and `avg_fare = NULL` (not a fabricated zero fare). This makes `LAG(..., 7)` correspond to seven calendar days rather than seven possibly nonconsecutive observations.

In [9]:
run_sql("""
DROP VIEW IF EXISTS v_t7_hw_taxi_features;
DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;

CREATE TABLE t7_hw_taxi_daily (
    pickup_zone_id INTEGER NOT NULL,
    ts_day DATE NOT NULL,
    trip_count INTEGER NOT NULL CHECK (trip_count >= 0),
    avg_fare NUMERIC(10, 2) CHECK (avg_fare IS NULL OR avg_fare > 0),
    total_distance NUMERIC(12, 2) NOT NULL CHECK (total_distance >= 0),
    PRIMARY KEY (pickup_zone_id, ts_day)
);

WITH daily_agg AS (
    SELECT
        "PULocationID"::INTEGER AS pickup_zone_id,
        tpep_pickup_datetime::TIMESTAMP::DATE AS ts_day,
        COUNT(*)::INTEGER AS trip_count,
        AVG(fare_amount)::NUMERIC(10,2) AS avg_fare,
        SUM(trip_distance)::NUMERIC(12,2) AS total_distance
    FROM nyc_taxi_raw
    GROUP BY "PULocationID"::INTEGER, tpep_pickup_datetime::TIMESTAMP::DATE
),
zones AS (
    SELECT DISTINCT pickup_zone_id FROM daily_agg
),
calendar_spine AS (
    SELECT
        z.pickup_zone_id,
        gs.day::DATE AS ts_day
    FROM zones AS z
    CROSS JOIN generate_series(
        DATE '2024-03-01',
        DATE '2024-03-31',
        INTERVAL '1 day'
    ) AS gs(day)
)
INSERT INTO t7_hw_taxi_daily (
    pickup_zone_id, ts_day, trip_count, avg_fare, total_distance
)
SELECT
    cs.pickup_zone_id,
    cs.ts_day,
    COALESCE(a.trip_count, 0),
    a.avg_fare,
    COALESCE(a.total_distance, 0)
FROM calendar_spine AS cs
LEFT JOIN daily_agg AS a
    ON a.pickup_zone_id = cs.pickup_zone_id
   AND a.ts_day = cs.ts_day;
""")

print('Daily aggregation and calendar spine created.')

Daily aggregation and calendar spine created.


In [10]:
display(run_query("""
SELECT
    COUNT(*) AS n_zone_days,
    COUNT(DISTINCT pickup_zone_id) AS n_zones,
    COUNT(DISTINCT ts_day) AS n_calendar_days,
    COUNT(*) FILTER (WHERE trip_count = 0) AS missing_zone_days,
    MIN(ts_day) AS first_day,
    MAX(ts_day) AS last_day
FROM t7_hw_taxi_daily;
"""))

display(run_query("""
SELECT pickup_zone_id, COUNT(*) AS n_days, SUM(trip_count) AS total_trips
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY total_trips DESC
LIMIT 10;
"""))

,n_zone_days,n_zones,n_calendar_days,missing_zone_days,first_day,last_day
0,7595,245,31,2353,2024-03-01,2024-03-31


,pickup_zone_id,n_days,total_trips
0,161,31,19812
1,237,31,18944
2,132,31,18777
3,236,31,17652
4,162,31,15181
5,230,31,14776
6,186,31,14170
7,142,31,13943
8,138,31,13169
9,239,31,12503


### 1.5. Local Time vs TIMESTAMPTZ

The TLC timestamps are local New York times. For an actual event timestamp, `AT TIME ZONE 'America/New_York'` converts the timestamp to a `TIMESTAMPTZ` instant. Only 1,000 rows are used.

In [11]:
run_sql("""
DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;

CREATE TABLE t7_hw_taxi_trips_typed AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    "DOLocationID"::INTEGER AS dropoff_zone_id,
    (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York')
        AS pickup_at,
    (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York')
        AS dropoff_at,
    trip_distance::NUMERIC(10,2) AS trip_distance,
    fare_amount::NUMERIC(10,2) AS fare_amount
FROM nyc_taxi_raw
ORDER BY tpep_pickup_datetime
LIMIT 1000;
""")

display(run_query("""
SELECT pickup_at, pg_typeof(pickup_at)::TEXT AS pickup_type
FROM t7_hw_taxi_trips_typed
LIMIT 5;
"""))

,pickup_at,pickup_type
0,2024-03-01 05:00:00+00:00,timestamp with time zone
1,2024-03-01 05:00:16+00:00,timestamp with time zone
2,2024-03-01 05:00:29+00:00,timestamp with time zone
3,2024-03-01 05:00:51+00:00,timestamp with time zone
4,2024-03-01 05:00:57+00:00,timestamp with time zone


## Task 2. Time-Series ML Features

All window functions partition by `pickup_zone_id`. The calendar spine makes row-based lags equivalent to calendar-day lags inside March. We use retrospective 7-day and 30-day windows that include the current day, as requested in the homework.

### 2.1. Lag Features: 1, 7 and 30 Days

In [12]:
query = """
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day DESC
LIMIT 20;
"""
display(run_query(query))

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-03-31,0,0,1,0.0
1,1,2024-03-30,0,0,0,NaN
2,1,2024-03-29,0,1,1,NaN
3,1,2024-03-28,1,1,2,NaN
4,1,2024-03-27,1,2,0,NaN
5,1,2024-03-26,2,0,0,NaN
6,1,2024-03-25,0,1,0,NaN
7,1,2024-03-24,1,0,0,NaN
8,1,2024-03-23,0,1,2,NaN
9,1,2024-03-22,1,2,1,NaN


**Note:** The first 30 days of each zone have no prior 30-day observation within the March-only dataset, so `trips_lag_30d` is `NULL` until March 31. This means it is **missing historical context**, not zero trips.

### 2.2. Rolling 7-Day Average Fare

In [13]:
query = """
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*) OVER w7 AS calendar_days_in_window,
    COUNT(avg_fare) OVER w7 AS observed_days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day DESC
LIMIT 20;
"""
display(run_query(query))

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,calendar_days_in_window,observed_days_in_window
0,1,2024-03-31,NaN,97.833333,7,3
1,1,2024-03-30,NaN,79.625000,7,4
2,1,2024-03-29,NaN,79.625000,7,4
3,1,2024-03-28,106.00,83.300000,7,5
4,1,2024-03-27,100.00,79.410000,7,5
5,1,2024-03-26,87.50,74.262500,7,4
6,1,2024-03-25,NaN,69.850000,7,3
7,1,2024-03-24,25.00,69.850000,7,3
8,1,2024-03-23,NaN,92.275000,7,2
9,1,2024-03-22,98.00,79.733333,7,3


### 2.3. Rolling 30-Day Fare Volatility

In [14]:
query = """
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(avg_fare) OVER w30 AS observed_days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day DESC
LIMIT 20;
"""
display(run_query(query))

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,observed_days_in_window
0,1,2024-03-31,NaN,29.080818,10
1,1,2024-03-30,NaN,29.080818,10
2,1,2024-03-29,NaN,29.080818,10
3,1,2024-03-28,106.00,29.080818,10
4,1,2024-03-27,100.00,30.212747,9
5,1,2024-03-26,87.50,31.906277,8
6,1,2024-03-25,NaN,34.455753,7
7,1,2024-03-24,25.00,34.455753,7
8,1,2024-03-23,NaN,23.776648,6
9,1,2024-03-22,98.00,23.776648,6


### 2.4. Expanding Cumulative Trips

In [15]:
query = """
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day DESC
LIMIT 20;
"""
display(run_query(query))

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-03-31,0,13,0.419355
1,1,2024-03-30,0,13,0.433333
2,1,2024-03-29,0,13,0.448276
3,1,2024-03-28,1,13,0.464286
4,1,2024-03-27,1,12,0.444444
5,1,2024-03-26,2,11,0.423077
6,1,2024-03-25,0,9,0.360000
7,1,2024-03-24,1,9,0.375000
8,1,2024-03-23,0,8,0.347826
9,1,2024-03-22,1,8,0.363636


### 2.5. Combined Feature View

In [16]:
run_sql("""
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    avg_fare,
    total_distance,
    LAG(trip_count, 1)  OVER w_lag AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w_lag AS trips_lag_7d,
    LAG(trip_count, 30) OVER w_lag AS trips_lag_30d,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    SUM(trip_count) OVER w_exp AS cumulative_trips,
    AVG(trip_count) OVER w_exp AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW
    w_lag AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
    ),
    w7 AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day::TIMESTAMP
        RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
    ),
    w30 AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day::TIMESTAMP
        RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
    ),
    w_exp AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    );
""")

display(run_query("""
SELECT *
FROM v_t7_hw_taxi_features
WHERE trips_lag_30d IS NOT NULL
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
"""))

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,1,2024-03-31,0,NaN,0.00,0,1,0,97.833333,29.080818,13,0.419355
1,3,2024-03-31,0,NaN,0.00,1,0,0,27.925000,17.768664,15,0.483871
2,4,2024-03-31,49,17.58,152.17,58,51,24,18.180000,2.458562,874,28.193548
3,6,2024-03-31,0,NaN,0.00,0,0,0,NaN,0.000000,2,0.064516
4,7,2024-03-31,25,25.02,136.35,10,11,4,19.434286,7.001742,275,8.870968
5,8,2024-03-31,0,NaN,0.00,0,0,0,NaN,41.931432,2,0.064516
6,9,2024-03-31,1,24.50,5.10,0,0,0,32.000000,24.988998,6,0.193548
7,10,2024-03-31,8,42.24,85.62,6,8,5,57.074286,17.811422,165,5.322581
8,11,2024-03-31,0,NaN,0.00,0,1,0,23.796667,17.635957,10,0.322581
9,12,2024-03-31,7,26.10,36.43,8,5,2,25.270000,7.171205,141,4.548387


### 2.6. Feature Validation

In [17]:
display(run_query("""
SELECT
    COUNT(*) AS n_feature_rows,
    COUNT(DISTINCT (pickup_zone_id, ts_day)) AS n_unique_zone_days,
    COUNT(*) FILTER (WHERE trips_lag_30d IS NOT NULL)
        AS n_rows_with_30d_history,
    COUNT(*) FILTER (WHERE trip_count < 0) AS negative_trip_counts,
    COUNT(*) FILTER (WHERE rolling_avg_fare_7d < 0)
        AS negative_rolling_fares
FROM v_t7_hw_taxi_features;
"""))

,n_feature_rows,n_unique_zone_days,n_rows_with_30d_history,negative_trip_counts,negative_rolling_fares
0,7595,7595,245,0,0


- Each zone has a complete March calendar (`31` rows per observed pickup zone), so `LAG(1)`, `LAG(7)` and `LAG(30)` represent calendar days instead of missing-row offsets.
- The 7-day `RANGE` frame includes the current date and six preceding dates; the 30-day frame includes the current date and 29 preceding dates.
- Missing zone-days have zero trips, but their average fares remain `NULL`; PostgreSQL `AVG()` and `STDDEV_SAMP()` ignore those `NULL` fares.
- `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` computes cumulative and expanding demand metrics.
- **ML leakage consideration:** current-day fares and trips are not available before the day finishes. These inclusive rolling values suit completed-day analysis. For a model forecasting a future day, shift the relevant features to use only information available at prediction time.
- With only March 2024 loaded, the first 30 days lack prior-month context for `lag_30d`. For production features, load February history as well.

# Task 3. JSONB Event Log

### 3.1. Create and Populate Event Log

In [18]:
query = """
DROP TABLE IF EXISTS t7_hw_checkouts_norm;
DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    user_id INTEGER NOT NULL,
    event_ts TIMESTAMPTZ NOT NULL,
    event_type TEXT NOT NULL CHECK (
        event_type IN (
            'page_view', 'click', 'add_to_cart',
            'checkout', 'recommendation_shown'
        )
    ),
    payload JSONB NOT NULL
);

INSERT INTO t7_hw_events (
    user_id, event_ts, event_type, payload
)
SELECT
    ((n * 37 + n / 5) % 1000) + 1,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL,
    CASE n % 5
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE 'recommendation_shown'
    END,
    CASE n % 5
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10,2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10,2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;
"""

run_sql(query)

In [19]:
display(run_query("""
SELECT
    event_type,
    COUNT(*) AS n_events,
    COUNT(DISTINCT user_id) AS n_users
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;
"""))

,event_type,n_events,n_users
0,add_to_cart,10000,500
1,checkout,10000,500
2,click,10000,500
3,page_view,10000,500
4,recommendation_shown,10000,500


### 3.2. JSONB Containment

Business Question: Which checkout events used card payments?

In [20]:
query = """
SELECT
    event_id,
    user_id,
    event_ts,
    payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> '{"payment_method": "card"}'::JSONB
ORDER BY event_id
LIMIT 10;
"""

display(run_query(query))

,event_id,user_id,event_ts,payload
0,3,112,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,18,670,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
2,33,228,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
3,48,786,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items..."
4,63,344,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'item..."
5,78,902,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'item..."
6,93,460,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'item..."
7,108,18,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'item..."
8,123,576,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'item..."
9,138,134,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'item..."


### 3.3. JSONB Key Existence

Business Question: Which events contain product tags, and which contain recommendation experiment metadata?

In [21]:
query = """
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
ORDER BY event_id
LIMIT 10;
"""

display(run_query(query))

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3..."
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quanti..."
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3..."
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity':..."
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quanti..."
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3..."
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity':..."
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quanti..."


In [22]:
query = """
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
ORDER BY event_id
LIMIT 10;
"""

display(run_query(query))

,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


### 3.4. JSONPath Queries

Business Question: Which products were recommended, and which checkout events have a total above 300?

In [23]:
query = """
SELECT
    event_id,
    jsonb_path_query(
        payload,
        '$.recommendations[*]'
    ) AS recommended_product_id
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
ORDER BY event_id
LIMIT 15;
"""

display(run_query(query))

,event_id,recommended_product_id
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [24]:
query = """
SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10,2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(
      payload,
      '$.total ? (@ > 300)'
  )
ORDER BY event_id
LIMIT 10;
"""

display(run_query(query))

,event_id,order_id,total
0,253,o_100253,303.0
1,258,o_100258,308.0
2,263,o_100263,313.0
3,268,o_100268,318.0
4,273,o_100273,323.0
5,278,o_100278,328.0
6,283,o_100283,333.0
7,288,o_100288,338.0
8,293,o_100293,343.0
9,298,o_100298,348.0


# Task 4. GIN Index and EXPLAIN

### 4.1. Prepare EXPLAIN show helper function

In [25]:
def show_explain(query):
    explain_query = (
        "EXPLAIN (ANALYZE, BUFFERS)\n"
        + query.strip().rstrip(";")
    )

    result = run_query(explain_query)
    plan = result.iloc[:, 0].astype(str).tolist()

    print("\n".join(plan))
    return plan

### 4.2. Containment Before GIN

In [26]:
run_sql("""
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;
""")

containment_query = """
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> '{"payment_method": "card"}'::JSONB
"""

gin_before_plan = show_explain(containment_query)

Aggregate  (cost=1670.39..1670.40 rows=1 width=8) (actual time=14.712..14.714 rows=1 loops=1)
  Buffers: shared hit=1038
  ->  Seq Scan on t7_hw_events  (cost=0.00..1663.00 rows=2957 width=0) (actual time=0.010..14.452 rows=3334 loops=1)
        Filter: (payload @> '{"payment_method": "card"}'::jsonb)
        Rows Removed by Filter: 46666
        Buffers: shared hit=1038
Planning:
  Buffers: shared hit=10
Planning Time: 0.154 ms
Execution Time: 14.742 ms


### 4.3. Create GIN Index

In [27]:
run_sql("""
CREATE INDEX idx_t7_events_payload_gin_pathops
ON t7_hw_events
USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;
""")

In [28]:
display(run_query("""
SELECT
    indexname,
    pg_size_pretty(
        pg_relation_size(indexname::regclass)
    ) AS index_size
FROM pg_indexes
WHERE tablename = 't7_hw_events'
ORDER BY indexname;
"""))

,indexname,index_size
0,idx_t7_events_payload_gin_pathops,1144 kB
1,t7_hw_events_pkey,1112 kB


### 4.4. Containment After GIN

In [29]:
gin_after_plan = show_explain(containment_query)

Aggregate  (cost=1115.03..1115.04 rows=1 width=8) (actual time=4.862..4.863 rows=1 loops=1)
  Buffers: shared hit=1041
  ->  Bitmap Heap Scan on t7_hw_events  (cost=32.52..1107.61 rows=2967 width=0) (actual time=0.965..4.594 rows=3334 loops=1)
        Recheck Cond: (payload @> '{"payment_method": "card"}'::jsonb)
        Heap Blocks: exact=1037
        Buffers: shared hit=1041
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..31.78 rows=2967 width=0) (actual time=0.596..0.597 rows=3334 loops=1)
              Index Cond: (payload @> '{"payment_method": "card"}'::jsonb)
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=16
Planning Time: 0.908 ms
Execution Time: 4.918 ms


### 4.5. Expression Equality Before B-tree

In [30]:
device_query = """
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
"""

btree_before_plan = show_explain(device_query)

Aggregate  (cost=1788.62..1788.63 rows=1 width=8) (actual time=11.847..11.848 rows=1 loops=1)
  Buffers: shared hit=1038
  ->  Seq Scan on t7_hw_events  (cost=0.00..1788.00 rows=250 width=0) (actual time=0.016..11.310 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1038
Planning Time: 0.081 ms
Execution Time: 11.874 ms


### 4.6. Create Expression B-tree Index and Recheck

In [31]:
run_sql("""
CREATE INDEX idx_t7_events_device_btree
ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;
""")

btree_after_plan = show_explain(device_query)

Aggregate  (cost=1283.53..1283.54 rows=1 width=8) (actual time=5.155..5.157 rows=1 loops=1)
  Buffers: shared hit=1037 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=97.74..1262.41 rows=8445 width=0) (actual time=0.792..4.361 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1037
        Buffers: shared hit=1037 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..95.63 rows=8445 width=0) (actual time=0.592..0.592 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=19 read=1
Planning Time: 0.254 ms
Execution Time: 5.189 ms


The GIN index reduced JSONB containment query execution time from 14.742 ms to 4.918 ms. PostgreSQL switched from a sequential scan to Bitmap Index Scan and Bitmap Heap Scan. The query returned 3,334 matching events.

The expression B-tree index reduced device equality query time from 11.874 ms to 5.189 ms. PostgreSQL also switched to bitmap index access, returning 8,333 matching events. The estimated row count became much closer to the actual count after index creation and ANALYZE.

Both indexes improved performance in this experiment, although buffer usage remained relatively high because matching rows were spread across many table pages. Indexes also require storage and add overhead to data modifications. The measured speedups should not be treated as guaranteed performance improvements for other workloads.

# Task 5. JSONB vs Normalized

### 5.1. Create Normalized Checkout Table

In [32]:
run_sql("""
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id BIGINT PRIMARY KEY
        REFERENCES t7_hw_events(event_id),
    user_id INTEGER NOT NULL,
    event_ts TIMESTAMPTZ NOT NULL,
    order_id TEXT NOT NULL,
    total NUMERIC(10,2) NOT NULL,
    items_count INTEGER NOT NULL
        CHECK (items_count > 0),
    payment_method TEXT NOT NULL
        CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id, user_id, event_ts, order_id,
    total, items_count, payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id',
    (payload ->> 'total')::NUMERIC(10,2),
    (payload ->> 'items_count')::INTEGER,
    payload ->> 'payment_method'
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;
""")

In [33]:
display(run_query("""
SELECT
    COUNT(*) AS n_checkouts,
    COUNT(DISTINCT payment_method) AS n_payment_methods
FROM t7_hw_checkouts_norm;
"""))

,n_checkouts,n_payment_methods
0,10000,3


### 5.2. JSONB Query EXPLAIN

Business Question: What is the average checkout total and number of checkouts for each payment method?

In [34]:
jsonb_checkout_query = """
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3
"""

jsonb_checkout_plan = show_explain(jsonb_checkout_query)

display(run_query(jsonb_checkout_query))

Limit  (cost=1992.73..1992.74 rows=3 width=72) (actual time=33.097..33.100 rows=3 loops=1)
  Buffers: shared hit=1038
  ->  Sort  (cost=1992.73..2006.59 rows=5544 width=72) (actual time=33.094..33.096 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1038
        ->  HashAggregate  (cost=1837.91..1921.07 rows=5544 width=72) (actual time=33.044..33.083 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1038
              ->  Seq Scan on t7_hw_events  (cost=0.00..1687.99 rows=9995 width=132) (actual time=0.016..23.021 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1038
Planning:
  Buffers: shared hit=29 read=1
Planning Time: 0.312 ms
Execution Time: 33.172

,payment_method,avg_total,n
0,paypal,300.524752,3333
1,card,300.500000,3334
2,crypto,300.475248,3333


### 5.3. Normalized Query EXPLAIN

In [35]:
normalized_checkout_query = """
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3
"""

normalized_checkout_plan = show_explain(normalized_checkout_query)

display(run_query(normalized_checkout_query))

Limit  (cost=279.06..279.07 rows=3 width=46) (actual time=10.080..10.082 rows=3 loops=1)
  Buffers: shared hit=104
  ->  Sort  (cost=279.06..279.07 rows=3 width=46) (actual time=10.078..10.079 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=104
        ->  HashAggregate  (cost=279.00..279.04 rows=3 width=46) (actual time=10.064..10.067 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=104
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..204.00 rows=10000 width=11) (actual time=0.009..1.505 rows=10000 loops=1)
                    Buffers: shared hit=104
Planning:
  Buffers: shared hit=3
Planning Time: 0.140 ms
Execution Time: 10.121 ms


,payment_method,avg_total,n
0,paypal,300.524752,3333
1,card,300.500000,3334
2,crypto,300.475248,3333


# Reflection

This homework combined daily NYC Taxi features with a synthetic JSONB event log. The calendar spine makes lags represent calendar days for each pickup zone. Rolling windows include the current day, so features used for forecasting at the start of a day would need shifting to prevent data leakage.

For JSONB containment, EXPLAIN ANALYZE showed 14.742 ms before indexing and 4.918 ms after creating a GIN index with `jsonb_path_ops`. The plan changed from Seq Scan to Bitmap Index Scan and Bitmap Heap Scan, while the estimated total cost fell from 1,670.40 to 1,115.04. GIN helps the `@>` operator find matching documents, but this operator class does not support key-existence queries such as `?`. For the expression `payload ->> 'device' = 'mobile'`, a dedicated B-tree index was more suitable: execution time decreased from 11.874 to 5.189 ms, with bitmap index access.

Indexes improve selective reads but take disk space and increase INSERT, UPDATE and DELETE costs because index entries must be maintained. The measured gains may vary with data distribution and caching.

The JSONB checkout aggregation took 33.172 ms with 1,038 shared buffer hits; the normalized version took 10.121 ms with 104 hits. Both returned identical counts and averages. I would use JSONB for diverse, evolving event payloads, and typed relational tables for stable checkout metrics and curated ML features. These tables differ in size, so the timing difference cannot be attributed solely to JSONB parsing.

To add a refund event, JSONB would allow a new payload structure, but the existing `event_type` CHECK constraint must also be updated. A normalized design would normally need a refund table, constraints, ingestion changes and possibly indexes. MongoDB could be considered for document-first workloads, but PostgreSQL is sufficient here.